In [6]:
import cv2
import mediapipe as mp
import os
import matplotlib.pyplot as plt 
import numpy as np
from scipy.signal import butter, lfilter,  filtfilt, savgol_filter
import pywt
from ultralytics import YOLO
from scipy.ndimage import zoom

# Função para criar o filtro passa-banda
def butter_bandpass(lowcut, highcut, fs, order=4):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return b, a

# Aplicação do filtro passa-banda

def apply_bandpass_filter_lfilter(data, lowcut, highcut, fs, order=4):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    filtered_data = lfilter(b, a, data)
    return filtered_data

def moving_average(signal, window_size):
    return np.convolve(signal, np.ones(window_size) / window_size, mode='valid')

# Remoção de baseline com ajuste de spline
def remove_baseline(signal, poly_order=2):
    x = np.arange(len(signal))
    coeffs = np.polyfit(x, signal, poly_order)  
    baseline = np.polyval(coeffs, x)     
    return signal - baseline  # Remove a baseline

def normalized_signal (signal):
    sig= np.array(signal) 
    signal_min = np.min(sig)
    signal_max = np.max(sig)
    normalized_signal = (sig - signal_min) / (signal_max - signal_min)
    return normalized_signal

model = YOLO("C:/Users/Douglas/Downloads/yolov8m_200e.pt")

# Inicializar FaceMesh
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1, refine_landmarks=True)

# Vídeo de entrada
cap = cv2.VideoCapture(r"C:\Users\Douglas\OneDrive - cefet-rj.br\Documentos\Material _ faculdade\Projeto IC\cohface\cohface\40\1\data.avi") 
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# ROI e parâmetros
resize_dim = (115, 50)
amplification_factor_x = 0.7 
amplification_factor_y = 0.7 

pontos_referencia = [103, 67, 104, 109, 10, 338, 297, 332, 333, 299, 337, 151, 108, 69, 104, 105, 66, 107, 9, 336, 296, 334]
conexoes = [
    (103, 67), (67, 109), (109, 10), (10, 338),
    (338, 297), (297, 332), (103, 105),
    (105, 66), (66, 107), (107, 9), (9, 336), (336, 296), (296, 334), (334, 332)
]

# Sinais
green_signal, R_channel, G_channel, B_channel = [], [], [], []
frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(frame, device='cuda')
    boxes = results[0].boxes.xyxy.cpu().numpy()

    if len(boxes) > 0:
        # Pegamos a primeira detecção
        x1, y1, x2, y2 = boxes[0].astype(int)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(frame_width, x2), min(frame_height, y2)

        face_roi = frame[y1:y2, x1:x2]
        frame_rgb_roi = cv2.cvtColor(face_roi, cv2.COLOR_BGR2RGB)

        result = face_mesh.process(frame_rgb_roi)

        if result.multi_face_landmarks:
            for face_landmarks in result.multi_face_landmarks:
                pontos = []
                for idx in pontos_referencia:
                    lm = face_landmarks.landmark[idx]
                    x = int(lm.x * (x2 - x1)) + x1
                    y = int(lm.y * (y2 - y1)) + y1
                    pontos.append((x, y))

                if pontos:
                    xs, ys = [p[0] for p in pontos], [p[1] for p in pontos]
                    x_min, x_max = max(0, min(xs)), min(frame_width, max(xs))
                    y_min, y_max = max(0, min(ys)), min(frame_height, max(ys))

                    roi = frame[y_min:y_max, x_min:x_max]
                    if roi.size > 0:
                        resized_roi = cv2.resize(roi, resize_dim)

                        R_channel.append(np.mean(resized_roi[:, :, 2]))
                        G_channel.append(np.mean(resized_roi[:, :, 1]))
                        B_channel.append(np.mean(resized_roi[:, :, 0]))
                        green_signal.append(np.mean(resized_roi[:, :, 1]))
      
    frame_count += 1

# Aplicar o filtro passa-banda no sinal verde
lowcut = 0.7  
highcut = 4.0  

window_size = 10  
poly_order = 2    # Ordem do polinômio para a baseline

# Conversão para numpy arrays
R_channel = np.array(R_channel)
G_channel = np.array(G_channel)
B_channel = np.array(B_channel)

# Cálculo dos sinais POS
X_POS = G_channel - 0.5 * (R_channel + B_channel)
Y_POS = 3 * G_channel - 2 * R_channel - B_channel

X_POS = (X_POS - np.mean(X_POS)) / np.std(X_POS)
Y_POS = (Y_POS - np.mean(Y_POS)) / np.std(Y_POS)

# Projeção do sinal POS
S_POS = X_POS + Y_POS

filtered_signal_2 = apply_bandpass_filter_lfilter(S_POS, lowcut, highcut, fps)

smoothed_signal_2 = moving_average(filtered_signal_2, window_size)
baseline_removed_signal_2 = remove_baseline (smoothed_signal_2, poly_order)

IPPG2 = baseline_removed_signal_2
IPPG2_normalized = normalized_signal (IPPG2)

filtered_signal_1 = apply_bandpass_filter_lfilter(green_signal, lowcut, highcut, fps)
smoothed_signal_1 = moving_average(filtered_signal_1, window_size)
baseline_removed_signal_1 = remove_baseline(smoothed_signal_1, poly_order)

IPPG1 = baseline_removed_signal_1   
IPPG1_normalized = normalized_signal(IPPG1)

# Encontrando IPPG3

IPPG3 = (IPPG1 + IPPG2) / 2

IPPG3_normalized = normalized_signal (IPPG3)

plt.figure(figsize=(12, 6))
plt.plot(IPPG1_normalized, label="Sinal Verde Filtrado e Suavizado", linewidth=2)
plt.xlabel("Segundos")
plt.ylabel("Amplitude Normalizada")
plt.title("Sinal IPPG1")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(IPPG2_normalized, label="Sinal POS Metodo", linewidth=2)
plt.xlabel("Segundos")
plt.ylabel("Amplitude Normalizada")
plt.title("Sinal IPPG2")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(IPPG3_normalized, label="Média IPPG1 e IPPG2", linewidth=2)
plt.xlabel("Segundos")
plt.ylabel("Amplitude Normalizada")
plt.title("Sinal IPPG3")
plt.legend()
plt.grid()
plt.show()

# CODIGO PARA RECORTAR O SINAL IPP3 
total_frames = 1000  
duracao_total = 40      
frs = total_frames / duracao_total

#print(f"Tamanho do sinal IPPG3_normalized: {len(IPPG3_normalized)}")

frames = np.arange(0, len(IPPG3_normalized))  
tempo = frames / frs

inicio_frame = int(1 * frs)
fim_frame = int(10 * frs)
sinal_recortado = IPPG3_normalized[inicio_frame:fim_frame + 1]
tempo_recortado = tempo[inicio_frame:fim_frame + 1]


def plot_and_save_wavelet(time, data, wavelet, input_video_path):
    # Calcula a CWT
    widths = np.geomspace(1, 1024, num=75)
    cwtmatr, freqs = pywt.cwt(
        data, widths, wavelet, sampling_period=np.diff(time).mean()
    )
    cwtmatr = np.abs(cwtmatr[:-1, :-1])
    print(cwtmatr.shape, cwtmatr.min(), cwtmatr.max())

    # Diretório do vídeo de entrada
    video_dir = os.path.dirname(input_video_path)
    video_name = os.path.splitext(os.path.basename(input_video_path))[0]

    # Salvar a matriz CWT
    np.save(os.path.join(video_dir, f"{video_name}_cwt.npy"), cwtmatr)

    # Gerar e salvar a imagem do espectrograma
    fig, ax = plt.subplots(figsize=(6, 6))
    pcm = ax.pcolormesh(time, freqs, cwtmatr, cmap='viridis', shading='auto')
    ax.set_axis_off()
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

    output_img_path = os.path.join(video_dir, f"{video_name}_espectrograma.png")
    fig.savefig(output_img_path, bbox_inches='tight', pad_inches=0)
    plt.close(fig)

    print(f"Arquivos salvos em: {video_dir}")
    return True

fs = 30

plot_and_save_wavelet(tempo_recortado, sinal_recortado, wavelet='cgau1',
                      input_video_path=r"C:\Users\Douglas\OneDrive - cefet-rj.br\Documentos\Material _ faculdade\Projeto IC\cohface\cohface\40\1\data.avi",
                      output_base_dir=r"C:\Users\Douglas\OneDrive - cefet-rj.br\Documentos\Material _ faculdade\Projeto IC\cohface\cohface\40\1")


AttributeError: module 'mediapipe' has no attribute 'solutions'

In [1]:
import cv2
import mediapipe as mp
import os
import matplotlib.pyplot as plt 
import numpy as np
from scipy.signal import butter, filtfilt
from scipy.signal import savgol_filter
import pywt
from ultralytics import YOLO
from scipy.ndimage import zoom

def process_video(input_video_path, output_base_dir):
    import mediapipe as mp
    # Função para criar o filtro passa-banda
    def butter_bandpass(lowcut, highcut, fs, order=4):
        nyquist = 0.5 * fs
        low = lowcut / nyquist
        high = highcut / nyquist
        b, a = butter(order, [low, high], btype='band')
        return b, a

    # Aplicação do filtro passa-banda
    def apply_bandpass_filter(data, lowcut, highcut, fs, order=4):
        b, a = butter_bandpass(lowcut, highcut, fs, order=order)
        filtered_data = filtfilt(b, a, data)
        return filtered_data

    def moving_average(signal, window_size):
        return np.convolve(signal, np.ones(window_size) / window_size, mode='valid')

    # Remoção de baseline com ajuste de spline
    def remove_baseline(signal, poly_order=2):
        x = np.arange(len(signal))
        coeffs = np.polyfit(x, signal, poly_order)  
        baseline = np.polyval(coeffs, x)     
        return signal - baseline  # Remove a baseline

    def normalized_signal(signal):
        sig= np.array(signal) 
        signal_min = np.min(sig)
        signal_max = np.max(sig)
        normalized_signal = (sig - signal_min) / (signal_max - signal_min)
        return normalized_signal

    model = YOLO("C:/Users/Douglas/Downloads/yolov8m_200e.pt")

# Inicializar FaceMesh
    mp_face_mesh = mp.solutions.face_mesh
    face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1, refine_landmarks=True)

    # Vídeo de entrada
    cap = cv2.VideoCapture(input_video_path) 
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    # ROI e parâmetros
    resize_dim = (115, 50)      
    amplification_factor_x = 0.7
    amplification_factor_y = 0.7  

    pontos_referencia = [103, 67, 104, 109, 10, 338, 297, 332, 333, 299, 337, 151, 108, 69, 104, 105, 66, 107, 9, 336, 296, 334]
    conexoes = [
        (103, 67), (67, 109), (109, 10), (10, 338),
        (338, 297), (297, 332), (103, 105),
        (105, 66), (66, 107), (107, 9), (9, 336), (336, 296), (296, 334), (334, 332)
    ]

    # Sinais
    green_signal, R_channel, G_channel, B_channel = [], [], [], []
    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = model.predict(frame, device='cuda')
        boxes = results[0].boxes.xyxy.cpu().numpy()

        if len(boxes) > 0:
            # Pegamos a primeira detecção
            x1, y1, x2, y2 = boxes[0].astype(int)
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(frame_width, x2), min(frame_height, y2)

            face_roi = frame[y1:y2, x1:x2]
            frame_rgb_roi = cv2.cvtColor(face_roi, cv2.COLOR_BGR2RGB)

            result = face_mesh.process(frame_rgb_roi)

            if result.multi_face_landmarks:
                for face_landmarks in result.multi_face_landmarks:
                    pontos = []
                    for idx in pontos_referencia:
                        lm = face_landmarks.landmark[idx]
                        x = int(lm.x * (x2 - x1)) + x1
                        y = int(lm.y * (y2 - y1)) + y1
                        pontos.append((x, y))

                    if pontos:
                        xs, ys = [p[0] for p in pontos], [p[1] for p in pontos]
                        x_min, x_max = max(0, min(xs)), min(frame_width, max(xs))
                        y_min, y_max = max(0, min(ys)), min(frame_height, max(ys))

                        roi = frame[y_min:y_max, x_min:x_max]
                        if roi.size > 0:
                            resized_roi = cv2.resize(roi, resize_dim)

                            R_channel.append(np.mean(resized_roi[:, :, 2]))
                            G_channel.append(np.mean(resized_roi[:, :, 1]))
                            B_channel.append(np.mean(resized_roi[:, :, 0]))
                            green_signal.append(np.mean(resized_roi[:, :, 1]))
        
        frame_count += 1

    # Aplicar o filtro passa-banda no sinal verde
    lowcut = 0.7  
    highcut = 4.0  
    window_size = 10  
    poly_order = 2    # Ordem do polinômio para a baseline

    # Conversão para numpy arrays
    R_channel = np.array(R_channel)
    G_channel = np.array(G_channel)
    B_channel = np.array(B_channel)

    # Cálculo dos sinais POS
    X_POS = G_channel - 0.5 * (R_channel + B_channel)
    Y_POS = 3 * G_channel - 2 * R_channel - B_channel

    X_POS = (X_POS - np.mean(X_POS)) / np.std(X_POS)
    Y_POS = (Y_POS - np.mean(Y_POS)) / np.std(Y_POS)

    # Projeção do sinal POS
    S_POS = X_POS + Y_POS
    filtered_signal_2 = apply_bandpass_filter(S_POS, lowcut, highcut, fps)

    smoothed_signal_2 = moving_average(filtered_signal_2, window_size)
    baseline_removed_signal_2 = remove_baseline(smoothed_signal_2, poly_order)

    IPPG2 = baseline_removed_signal_2
    IPPG2_normalized = normalized_signal(IPPG2)

    filtered_signal_1 = apply_bandpass_filter(green_signal, lowcut, highcut, fps)
    smoothed_signal_1 = moving_average(filtered_signal_1, window_size)
    baseline_removed_signal_1 = remove_baseline(smoothed_signal_1, poly_order)

    IPPG1 = baseline_removed_signal_1   
    IPPG1_normalized = normalized_signal(IPPG1)

    # Encontrando IPPG3
    IPPG3 = (IPPG1_normalized + IPPG2_normalized) / 2
    IPPG3_normalized = normalized_signal(IPPG3)

    # Recortar o sinal IPP3     
    fps = 25

    frames = np.arange(0, len(IPPG3_normalized))  
    tempo = frames / fps

    inicio_frame = int(1 * fps)
    fim_frame = int(10 * fps)
    sinal_recortado_ippg3 = IPPG3_normalized[inicio_frame:fim_frame + 1]
    tempo_recortado = tempo[inicio_frame:fim_frame + 1]

    tempo_recortado = tempo[inicio_frame:fim_frame + 1] 

    def plot_and_save_wavelet(time, data, wavelet, input_video_path):
        # Calcula a CWT
        widths = np.geomspace(1, 1024, num=75)
        cwtmatr, freqs = pywt.cwt(
            data, widths, wavelet, sampling_period=np.diff(time).mean()
        )
        cwtmatr = np.abs(cwtmatr[:-1, :-1])
        print(f"Forma original da CWT: {cwtmatr.shape}, min: {cwtmatr.min()} ,max: {cwtmatr.max()}")


        # Diretório do vídeo de entrada
        video_dir = output_base_dir
        video_name = os.path.splitext(os.path.basename(input_video_path))[0]

        np.save(os.path.join(video_dir, f"{video_name}_cwt.npy"), cwtmatr)

        # Gerar e salvar a imagem do espectrograma redimensionado
        fig, ax = plt.subplots(figsize=(6, 6))
        pcm = ax.imshow(cwtmatr, cmap='viridis', aspect='auto')
        ax.set_axis_off()
        plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

        output_img_path = os.path.join(video_dir, f"{video_name}_espectrograma.png")
        fig.savefig(output_img_path, bbox_inches='tight', pad_inches=0)
        plt.close(fig)

        print(f"Arquivos salvos em: {video_dir}")
        return True

    plot_and_save_wavelet(tempo_recortado, sinal_recortado_ippg3, wavelet='cgau1',
                        input_video_path= input_video_path)
    
    return True



In [ ]:
import os

def main():
    base_dir = r"C:\Users\Douglas\OneDrive - cefet-rj.br\Documentos\Material _ faculdade\Projeto IC\videos restantes"              # caminho da base
    output_dir = r"C:\Users\Douglas\OneDrive - cefet-rj.br\Documentos\Material _ faculdade\Projeto IC\VIPL-HR_CWT"      # nova pasta para salvar

    os.makedirs(output_dir, exist_ok=True)

    # Percorre pastas tipo p1-5, p6-10, etc.
    for group in os.listdir(base_dir):
        group_path = os.path.join(base_dir, group)

        if not os.path.isdir(group_path):
            continue

        # Percorre indivíduos (p1, p2, ...)
        for person in os.listdir(group_path):
            person_path = os.path.join(group_path, person)

            if not os.path.isdir(person_path):
                continue

            # 🔥 Apenas cenários desejados
            for scenario in ['v1', 'v7']:
                scenario_path = os.path.join(person_path, scenario)

                if not os.path.exists(scenario_path):
                    continue

                # source1, source2...
                for source in os.listdir(scenario_path):
                    video_path = os.path.join(scenario_path, source, "video.avi")

                    if not os.path.exists(video_path):
                        print(f"Vídeo não encontrado: {video_path}")
                        continue

                    # Criar pasta de saída espelhada
                    save_path = os.path.join(output_dir, group, person, scenario, source)
                    os.makedirs(save_path, exist_ok=True)

                    print(f"Processando: {video_path}")

                    try:
                        # 🔥 CHAMADA DA SUA FUNÇÃO (sem alterar nada)
                        process_video(video_path, save_path)

                    except Exception as e:
                        print(f"Erro ao processar {video_path}: {e}")


if __name__ == "__main__":
    main()

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d


def compute_label_1_10(video_folder):
    time_path = os.path.join(video_folder, "time.txt")
    hr_path = os.path.join(video_folder, "gt_HR.csv")

    if not os.path.exists(time_path) or not os.path.exists(hr_path):
        return None

    # tempo (s)
    time = np.loadtxt(time_path) / 1000.0

    # HR
    hr = pd.read_csv(hr_path).values.squeeze()

    # alinhar HR ao tempo real
    time_hr = np.linspace(0, time[-1], len(hr))
    interp = interp1d(time_hr, hr, fill_value="extrapolate")
    hr_interp = interp(time)

    # 🔥 selecionar 1s a 10s
    mask = (time >= 1) & (time <= 10)
    hr_segment = hr_interp[mask]

    if len(hr_segment) == 0:
        return None

    return np.mean(hr_segment)


def generate_labels(dataset_root, original_root):
    for root, _, files in os.walk(dataset_root):

        if "cwt.npy" in files:  # sua amostra

            rel_path = os.path.relpath(root, dataset_root)
            original_folder = os.path.join(original_root, rel_path)

            hr_label = compute_label_1_10(original_folder)

            if hr_label is None:
                print(f"Erro: {original_folder}")
                continue

            label_path = os.path.join(root, "label.txt")

            with open(label_path, "w") as f:
                f.write(str(hr_label))

            print(f"OK: {label_path}")
            
            
dataset_root = "C:/VIPL_HR_CWT"
original_root = "C:/VIPL-HR"

generate_labels(dataset_root, original_root)

In [1]:
import os
import numpy as np
import pandas as pd


def compute_label_no_time(video_folder):
    hr_path = os.path.join(video_folder, "gt_HR.csv")

    if not os.path.exists(hr_path):
        print(f"[ERRO] HR não encontrado: {video_folder}")
        return None

    try:
        hr = pd.read_csv(hr_path).values.squeeze()

        # 🔥 assumir HR amostrado uniformemente
        fps_hr = 1  # 1 valor por segundo (VIPL-HR geralmente)

        time = np.arange(len(hr)) / fps_hr

        # janela fixa 1–10s
        mask = (time >= 1) & (time <= 10)
        hr_segment = hr[mask]

        if len(hr_segment) == 0:
            print(f"[ERRO] Segmento vazio: {video_folder}")
            return None

        return np.mean(hr_segment)

    except Exception as e:
        print(f"[ERRO] Falha em {video_folder}: {e}")
        return None


def generate_missing_labels(cwt_root, vipl_root):

    total = 0
    criados = 0
    ignorados = 0
    erros = 0

    for root, _, files in os.walk(cwt_root):

        # 🔍 só processa se tiver CWT
        if not any(f.endswith(".npy") for f in files):
            continue

        total += 1

        label_path = os.path.join(root, "label.txt")

        # ✅ já existe → NÃO faz nada
        if os.path.exists(label_path):
            ignorados += 1
            continue

        # 🔗 mapear para VIPL-HR
        rel_path = os.path.relpath(root, cwt_root)
        original_folder = os.path.join(vipl_root, rel_path)

        if not os.path.exists(original_folder):
            print(f"[ERRO] Pasta não encontrada: {original_folder}")
            erros += 1
            continue

        hr_label = compute_label_no_time(original_folder)

        if hr_label is None:
            erros += 1
            continue

        # 💾 salvar
        with open(label_path, "w") as f:
            f.write(str(hr_label))

        print(f"[OK] Criado: {label_path}")
        criados += 1

    print("\n===== RESUMO =====")
    print(f"Total de amostras: {total}")
    print(f"Criados: {criados}")
    print(f"Ignorados (já existiam): {ignorados}")
    print(f"Erros: {erros}")
            
cwt_root = r"C:\Users\Douglas\Documents\VIPL_HR_IPPG"
vipl_root = r"C:\Users\Douglas\OneDrive - cefet-rj.br\Documentos\Material _ faculdade\Projeto IC\VIPL-HR"

generate_missing_labels(cwt_root, vipl_root)

[OK] Criado: C:\Users\Douglas\Documents\VIPL_HR_IPPG\p1-5\p1\v1\source1\label.txt
[OK] Criado: C:\Users\Douglas\Documents\VIPL_HR_IPPG\p1-5\p1\v1\source2\label.txt
[OK] Criado: C:\Users\Douglas\Documents\VIPL_HR_IPPG\p1-5\p1\v1\source3\label.txt
[OK] Criado: C:\Users\Douglas\Documents\VIPL_HR_IPPG\p1-5\p1\v2\source1\label.txt
[OK] Criado: C:\Users\Douglas\Documents\VIPL_HR_IPPG\p1-5\p1\v2\source2\label.txt
[OK] Criado: C:\Users\Douglas\Documents\VIPL_HR_IPPG\p1-5\p1\v2\source3\label.txt
[OK] Criado: C:\Users\Douglas\Documents\VIPL_HR_IPPG\p1-5\p1\v3\source1\label.txt
[OK] Criado: C:\Users\Douglas\Documents\VIPL_HR_IPPG\p1-5\p1\v3\source2\label.txt
[OK] Criado: C:\Users\Douglas\Documents\VIPL_HR_IPPG\p1-5\p1\v3\source3\label.txt
[OK] Criado: C:\Users\Douglas\Documents\VIPL_HR_IPPG\p1-5\p1\v4\source1\label.txt
[OK] Criado: C:\Users\Douglas\Documents\VIPL_HR_IPPG\p1-5\p1\v4\source2\label.txt
[OK] Criado: C:\Users\Douglas\Documents\VIPL_HR_IPPG\p1-5\p1\v4\source3\label.txt
[OK] Criado: C:\